In [1]:
#import libraries and data

import pandas as pd
from tqdm import tqdm
import numpy as np
from datetime import datetime, date, timedelta

path = "C:\\Users\\oscar\\big_ahh_files\\DW model\\game_fact.csv"
gfact = pd.read_csv(path)

path = "C:\\Users\\oscar\\big_ahh_files\\DW model\\daily_fact.csv"
dfact = pd.read_csv(path)

path = "C:\\Users\\oscar\\big_ahh_files\\DW model\\IL_update_fact.csv"
ifact = pd.read_csv(path)

C:\Users\oscar\AppData\Local\Temp\ipykernel_45704\89236666.py:9: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  gfact = pd.read_csv(path)


In [3]:
"""
I problem I noticed in the data is that the original daily fact table incorrectly classifies a player as injured.
On days of "injury," a player will play NBA regular season games. This blocks demonstrates this effect with three players.
The following blocks will address this problem by inferring missing records of updates to the NBA injury list from the
games that players play and make a more accurate daily fact table.
"""

# function identifies matches in two iterables and returns a list of matching values

def show_matches(l1, l2):
    
    matches = []
    
    for val in l1:
        if val in l2:
            matches.append(val)

    return matches

# given daily fact DataFrame (I will use the original and new one), the function prints the the dates where the player was injured and played a game

def games_played_while_injured(pk, daily_fact_df):

    # get a series of dates where the player is injured
    
    t1 = daily_fact_df.loc[daily_fact_df['player_key'] == pk, :]
    t1 = t1.loc[t1['injured?'] == True, 'date']

    # get a series of the dates where the player played an NBA regular season game
    
    t2 = gfact.loc[gfact['player_key'] == pk, :]
    t2 = t2.loc[t2['did_not_play'] == False, 'game_date']

    # find the matching dates between dates of injury and dates of play and print the matching dates
    
    matches = show_matches(list(t1), list(t2))
    print('Games were played during injury on these dates: \n', matches)

# audit the original daily fact table with three players

games_played_while_injured(2037, dfact) # many overlapping days
games_played_while_injured(1628427, dfact) # many overlapping days
games_played_while_injured(203995, dfact) # no overlapping day

Games were played during injury on these dates: 
 ['2017-10-18', '2017-10-20', '2017-10-22', '2017-10-24', '2017-10-25', '2017-10-27', '2017-10-30', '2017-11-01', '2017-11-04', '2017-11-05', '2017-11-08', '2017-11-11', '2017-11-13', '2017-11-15', '2017-11-17', '2017-11-19', '2017-11-20', '2017-11-22', '2017-11-24', '2017-11-26', '2017-11-28', '2017-11-29', '2017-12-01', '2017-12-03', '2017-12-04', '2017-12-06', '2017-12-10', '2017-12-12', '2017-12-14', '2017-12-16', '2017-12-18', '2017-12-20', '2017-12-23', '2017-12-25', '2017-12-27', '2017-12-28', '2017-12-31', '2018-01-01', '2018-01-03', '2018-01-05', '2018-01-06', '2018-01-08', '2018-01-10', '2018-01-12', '2018-01-14', '2018-01-16', '2018-01-18']
Games were played during injury on these dates: 
 ['2020-12-28', '2021-01-03', '2021-01-09', '2021-01-10', '2021-01-19', '2021-01-27', '2021-01-29', '2021-01-31', '2021-02-04', '2021-02-06', '2021-02-08', '2021-02-14', '2021-02-16', '2021-02-19', '2021-02-21', '2021-02-25', '2021-02-27', '2

In [5]:
#turn relevant columns into lists for faster processing

gfact_pk = list(gfact['player_key'])
gfact_date = list(gfact['game_date'])
gfact_dnp = list(gfact['did_not_play'])

ifact_pk = list(ifact['player_key'])
ifact_date = list(ifact['date'])
ifact_ra = list(ifact['relinquished/acquired'])

dfact_pk = list(dfact['player_key'])
dfact_date = list(dfact['date'])
dfact_status = list(dfact['injured?'])

In [7]:
#iterative fixing

pks = list(dfact['player_key'].unique())

for pk in tqdm(pks):

    ##########
    # STEP 1: DEFINE AND CREATE LISTS FROM INFORMATION THAT WILL BE USED IN THE SCRIPT
    ##########

    # create list of indices where the row in gfact belongs to the current player

    gfact_indices = []
    for i in range(len(gfact_pk)):
        if gfact_pk[i] == pk:
            gfact_indices.append(i)

    # isolate information belonging exclusively to the player from the gfact lists

    curr_gfact_date = []
    curr_gfact_dnp = []
    for i in gfact_indices:
        curr_gfact_date.append(gfact_date[i])
        curr_gfact_dnp.append(gfact_dnp[i])
    
    # create list of indices where the row in ifact belongs to the current player

    ifact_indices = []
    for i in range(len(ifact_pk)):
        if ifact_pk[i] == pk:
            ifact_indices.append(i)

    # isolate information belonging exclusively to the player from the ifact lists

    curr_ifact_date = []
    curr_ifact_ra = []
    for i in ifact_indices:
        curr_ifact_date.append(ifact_date[i])
        curr_ifact_ra.append(ifact_ra[i])

    # create list of indices where the row  in dfact belongs to the current player

    dfact_indices = []
    for i in range(len(dfact_pk)):
        if dfact_pk[i] == pk:
            dfact_indices.append(i)

    # isolate information belonging exclusively to the player from the dfact lists

    curr_dfact_date = []
    curr_dfact_status = []
    for i in dfact_indices:
        curr_dfact_date.append(dfact_date[i])
        curr_dfact_status.append(dfact_status[i])


    ##########
    # STEP 2: ITERATE THROUGH DFACT INFO AND GROUP A PLAYERS DAYS INTO STRETCHES BASED ON CHANGES TO THE INJURY LIST
    ##########

    stretches = []
    current_stretch = []
    
    for i in dfact_indices:
        """
        A stretch is a continuous period of days starting with a record in ifact where the player was removed and
        put on the injury list. The day of the record if the first day in the stretch. When there is a new record
        from the injury list, the current stretch is closed and saved and a new stretch is started 
        """
        
        current_date = dfact_date[i]

        # if change to the injury list, end current_stretch, append to stretches, and start new current_stretch

        if current_date in curr_ifact_date:
            
            stretches.append(current_stretch)
            current_stretch = [] # clear current_stretch
            current_stretch.append(i) # start new stretch

        # if on the last date, add the date to the current_stretch and append to stretches
        
        elif i == dfact_indices[-1]:
            current_stretch.append(i)
            stretches.append(current_stretch)

        # otherwise, continue the current stretch
        
        else:
            current_stretch.append(i)

    # now that the stretches have been identified...

    ##########
    # STEP 3: ITERATE THROUGH EACH STRETCH AND RECORD THE INDICES OF ROWS THAT WHERE A PLAYER WAS INJURED BUT STARTED PLAYING GAMES
    ##########
    
    # iterate through each stretch and convert values if a player recorded as injured starts playing games

    indices_of_change = [] # indices where a player is recorded as injured but should not be because they were playing games...
    #this list will be used later to change the values of rows that state the player is injured when the player is actually healthy
    
    for stretch in stretches:
        
        contradiction_found = False # this will be flipped if the player plays a game while they are injured

        # skip stretches that are recording the player as healthy (injured? == False)

        if dfact_status[stretch[0]] == False:
            continue

        for i in stretch:

            # if on the first date of the stretch, skip, because their placement on the injury list...
            # could be to rest for a game of that day or result from an injury during play
            
            if i == stretch[0]: 
                continue

            if contradiction_found: # after the contradiction is found, all remaining indices of rows append to indices_of_change
                indices_of_change.append(i)

            current_date = dfact_date[i]
            
            # if the current date in the stretch falls on a game day for the player and the player played in that game...
            
            if current_date in curr_gfact_date:

                # given that the player had a game on the current date, this block finds whether the player played in that game

                dnp_status = True # default did_not_play status
                
                for j in gfact_indices:
                    if gfact_date[j] == current_date:
                        dnp_status = gfact_dnp[j]

                # given that the player had a game, if the player played in that game, the row should mark injured? as False
                
                if dnp_status == False:
                    
                    indices_of_change.append(i) # append the index of the row to indices_of_change
                    contradiction_found = True

    ##########
    # STEP 4: CORRECT THE INCORRECT ROWS
    ##########
    
    for i in indices_of_change:
        dfact_status[i] = False

100%|██████████| 1199/1199 [06:13<00:00,  3.21it/s]


In [9]:
# make a new DataFrame from the corrected information and verify that the new version of dfact is the same shape as the original

new_dfact = pd.DataFrame({'injured?':dfact_status})
new_dfact = pd.concat([dfact[['player_key', 'team_key', 'date', 'age', 'experience']], new_dfact], axis=1)

print('dfact: ', dfact.shape)
print('new_dfact: ', new_dfact.shape)

new_dfact.head()

dfact:  (1519374, 6)
new_dfact:  (1519374, 6)


,player_key,team_key,date,age,experience,injured?
0,1713,23,2017-09-17,14844,7019,True
1,1713,23,2017-09-18,14845,7020,True
2,1713,23,2017-09-19,14846,7021,True
3,1713,23,2017-09-20,14847,7022,True
4,1713,23,2017-09-21,14848,7023,True


In [11]:
# audit three players again to confirm that there is no overlap

games_played_while_injured(2037, new_dfact) # now no overlapping dates
games_played_while_injured(1628427, new_dfact) # now no overlapping dates
games_played_while_injured(203995, new_dfact) # still no overlapping dates

Games were played during injury on these dates: 
 []
Games were played during injury on these dates: 
 []
Games were played during injury on these dates: 
 []


In [13]:
new_dfact.to_csv('C:\\Users\\oscar\\big_ahh_files\\DW Model\\daily_fact(2).csv', index=False)